In [ ]:
!pip install pandas numpy matplotlib seaborn scikit-learn nltk openpyxl

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import nltk

from nltk.sentiment.vader import SentimentIntensityAnalyzer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from textblob import TextBlob

nltk.download('vader_lexicon')

In [ ]:
os.makedirs("visualizations", exist_ok=True)

In [ ]:
df = pd.read_excel("test.xlsx")

print("Dataset Shape:", df.shape)
print("Columns:", df.columns)

df.head()

In [ ]:
# Convert date
df['date'] = pd.to_datetime(df['date'])

# Rename 'from' to Employee_ID (clean name)
df = df.rename(columns={'from': 'Employee_ID'})

# Create Month column
df['Month'] = df['date'].dt.to_period('M')

TASK 1 –SENTIMENT LABELING

In [ ]:
from textblob import TextBlob

def get_sentiment(text):
    analysis = TextBlob(str(text))
    polarity = analysis.sentiment.polarity
    
    if polarity > 0:
        return "Positive"
    elif polarity < 0:
        return "Negative"
    else:
        return "Neutral"

# Apply sentiment to email body
df['Sentiment'] = df['body'].apply(get_sentiment)

# Optional: store polarity score (makes project stronger)
df['Polarity'] = df['body'].apply(lambda x: TextBlob(str(x)).sentiment.polarity)

# Map sentiment to numerical score
sentiment_map = {
    "Positive": 1,
    "Negative": -1,
    "Neutral": 0
}

df['Score'] = df['Sentiment'].map(sentiment_map)

df.head()

TASK 2 – EDA

In [ ]:
#Sentiment Distribution
plt.figure()
sns.countplot(x='Sentiment', data=df)
plt.title("Sentiment Distribution")
plt.savefig("visualizations/sentiment_distribution.png", bbox_inches='tight')
plt.show()

#Message Length
df['Message_Length'] = df['body'].apply(len)

plt.figure()
sns.histplot(df['Message_Length'], bins=30)
plt.title("Message Length Distribution")
plt.savefig("visualizations/message_length_distribution.png", bbox_inches='tight')
plt.show()

#Monthly Sentiment Trend
monthly_trend = df.groupby('Month')['Score'].mean()

plt.figure()
monthly_trend.plot()
plt.title("Average Monthly Sentiment")
plt.savefig("visualizations/monthly_sentiment_trend.png", bbox_inches='tight')
plt.show()

TASK 3 – MONTHLY SCORING

In [ ]:
monthly_scores = df.groupby(['Employee_ID','Month'])['Score'].sum().reset_index()

monthly_scores.head()

TASK 4 – EMPLOYEE RANKING

In [ ]:
# Top 3 Positive
top_positive = monthly_scores.sort_values(
    by=['Month','Score','Employee_ID'],
    ascending=[True, False, True]
).groupby('Month').head(3)

print("Top Positive Employees:")
top_positive



In [ ]:
# Top 3 Negative
top_negative = monthly_scores.sort_values(
    by=['Month','Score','Employee_ID'],
    ascending=[True, True, True]
).groupby('Month').head(3)

print("Top Negative Employees:")
top_negative

TASK 5 – FLIGHT RISK

In [ ]:
df = df.sort_values(by=['Employee_ID','date'])

df['Neg_Flag'] = (df['Sentiment'] == "Negative").astype(int)

rolling_neg = (
    df.set_index('date')
      .groupby('Employee_ID')['Neg_Flag']
      .rolling('30D')
      .sum()
      .reset_index()
)

flight_risk = rolling_neg[rolling_neg['Neg_Flag'] >= 4]['Employee_ID'].unique()

print("Flight Risk Employees:")
print(flight_risk)

TASK 6 – LINEAR REGRESSION

In [ ]:
monthly_features = df.groupby(['Employee_ID','Month']).agg({
    'Score': 'sum',
    'body': 'count',
    'Message_Length': 'mean'
}).reset_index()

monthly_features.columns = [
    'Employee_ID',
    'Month',
    'Sentiment_Score',
    'Message_Count',
    'Avg_Message_Length'
]

monthly_features.head()

X = monthly_features[['Message_Count','Avg_Message_Length']]
y = monthly_features['Sentiment_Score']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)

predictions = model.predict(X_test)

print("MSE:", mean_squared_error(y_test, predictions))
print("R2 Score:", r2_score(y_test, predictions))

plt.figure()
plt.scatter(y_test, predictions)
plt.xlabel("Actual Sentiment Score")
plt.ylabel("Predicted Sentiment Score")
plt.title("Model Performance")
plt.savefig("visualizations/model_performance.png", bbox_inches='tight')
plt.show()